# Dataset EDA

This notebook checks that the precomputed SHARP Doppler traces are available under `data/doppler_traces`. If the folder is missing, it downloads `doppler_traces.zip` from Google Drive and extracts it into the expected layout.

In [ ]:
from pathlib import Path
import shutil
import zipfile

DATA_DIR = Path("..").resolve() / "data"
DOPPLER_DIR = DATA_DIR / "doppler_traces"
ZIP_PATH = DATA_DIR / "doppler_traces.zip"
DRIVE_FILE_ID = "1vGHldHZVb9hQ1_YfFqZHqF8Ty1DQZEea"

expected_subdirs = [
    "S1a", "S1b", "S1c", "S2a", "S2b", "S3a",
    "S4a", "S4b", "S5a", "S6a", "S6b", "S7a",
]


def has_doppler_traces(path: Path) -> bool:
    return path.exists() and all((path / subdir).is_dir() for subdir in expected_subdirs)


def download_doppler_zip(output_path: Path) -> None:
    try:
        import gdown
    except ImportError as exc:
        raise ImportError(
            "Install notebook dependencies first: pip install -r ../requirements.txt"
        ) from exc

    output_path.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(id=DRIVE_FILE_ID, output=str(output_path), quiet=False, fuzzy=True)


def extract_doppler_zip(zip_path: Path, data_dir: Path) -> None:
    extract_tmp = data_dir / "_doppler_extract_tmp"
    if extract_tmp.exists():
        shutil.rmtree(extract_tmp)
    extract_tmp.mkdir(parents=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_tmp)

    extracted_doppler = extract_tmp / "doppler_traces"
    if not extracted_doppler.is_dir():
        matches = [p for p in extract_tmp.rglob("doppler_traces") if p.is_dir()]
        if not matches:
            raise FileNotFoundError("The zip file does not contain a doppler_traces folder.")
        extracted_doppler = matches[0]

    if DOPPLER_DIR.exists():
        shutil.rmtree(DOPPLER_DIR)
    shutil.move(str(extracted_doppler), str(DOPPLER_DIR))
    shutil.rmtree(extract_tmp)


if has_doppler_traces(DOPPLER_DIR):
    print(f"Doppler traces already available: {DOPPLER_DIR}")
else:
    print("Doppler traces missing. Downloading doppler_traces.zip...")
    download_doppler_zip(ZIP_PATH)
    print("Extracting into data/doppler_traces...")
    extract_doppler_zip(ZIP_PATH, DATA_DIR)
    if not has_doppler_traces(DOPPLER_DIR):
        raise RuntimeError(f"Extraction finished, but expected folders are missing under {DOPPLER_DIR}")
    print(f"Doppler traces ready: {DOPPLER_DIR}")

In [ ]:
import pickle
import matplotlib.pyplot as plt

path = DOPPLER_DIR / "S1a" / "S1a_C_stream_0.txt"

with open(path, "rb") as f:
    trace = pickle.load(f)

print(trace.shape)

plt.imshow(trace.T, aspect="auto", origin="lower", cmap="viridis")
plt.xlabel("time")
plt.ylabel("Doppler bin")
plt.colorbar(label="normalized power")
plt.show()